# 04.4 Mutable vs Immutable Objects

This is the most consequential distinction in the chapter. Whether a type can be
changed in place decides how it behaves when shared, passed to functions, used as
a dictionary key, or set as a default argument.

## Theory

### The definition

- **Mutable** — the object's value can change *while keeping the same identity*
- **Immutable** — the value can never change; any "change" creates a new object

The test is not whether you can write code that appears to modify it. The test is
whether `id()` stays the same.

### The two groups

<table>
<tr><th>Immutable</th><th>Mutable</th></tr>
<tr><td><code>int</code>, <code>float</code>, <code>complex</code>, <code>bool</code></td><td><code>list</code></td></tr>
<tr><td><code>str</code></td><td><code>dict</code></td></tr>
<tr><td><code>tuple</code></td><td><code>set</code></td></tr>
<tr><td><code>frozenset</code></td><td><code>bytearray</code></td></tr>
<tr><td><code>bytes</code></td><td>most custom classes</td></tr>
<tr><td><code>None</code></td><td></td></tr>
</table>

A useful pattern: the **built-in collections come in pairs** — `list`/`tuple`,
`set`/`frozenset`, `bytearray`/`bytes`. One mutable, one immutable.

### Why immutability exists

Three concrete reasons, not philosophy:

1. **Hashability.** Dictionary keys and set members must not change, or the
   container could never find them again. Only immutable objects are hashable.
2. **Safety when shared.** An immutable object can be shared freely — no caller
   can alter it behind your back.
3. **Optimisation.** Python can cache and intern immutable values (04.3) because
   sharing them is safe.

### The subtlety that catches people

A tuple is immutable — but that only means **the references it holds cannot be
replaced**. If a tuple contains a list, that list is still mutable.

Immutability is **shallow**.

In [ ]:
# Test any object by watching id() across an apparent modification.

def test_mutability(label, value, change):
    """Apply a change and report whether the identity survived."""
    identity_before = id(value)

    try:
        change(value)
        identity_after = id(value)
        if identity_after == identity_before:
            verdict = "MUTABLE - changed in place"
        else:
            verdict = "rebound to a new object"
    except (TypeError, AttributeError) as error:
        verdict = f"IMMUTABLE - {type(error).__name__}"

    print(f"   {label:<12} {verdict}")


print("Attempting an in-place change on each type:")
print("")

test_mutability("list", [1, 2], lambda v: v.append(3))
test_mutability("dict", {"a": 1}, lambda v: v.update({"b": 2}))
test_mutability("set", {1, 2}, lambda v: v.add(3))
test_mutability("bytearray", bytearray(b"ab"), lambda v: v.append(99))
test_mutability("tuple", (1, 2), lambda v: v.__setitem__(0, 9))
test_mutability("str", "ab", lambda v: v.__setitem__(0, "z"))
test_mutability("frozenset", frozenset([1]), lambda v: v.add(2))
test_mutability("int", 5, lambda v: v.__setitem__(0, 1))

## Watching identity during modification

The clearest demonstration: mutate a list and a string the same way, and compare
what happens to `id()`.

In [ ]:
# MUTABLE: the object changes, the identity does not.
shopping = ["bread", "milk"]
identity_before = id(shopping)

shopping.append("eggs")
shopping[0] = "sourdough"

print("LIST (mutable)")
print("   before id:", identity_before)
print("   after  id:", id(shopping))
print("   same object?", id(shopping) == identity_before)
print("   value now:", shopping)

# IMMUTABLE: every "change" produces a different object.
greeting = "hello"
identity_before = id(greeting)

greeting = greeting.upper()

print("")
print("STR (immutable)")
print("   before id:", identity_before)
print("   after  id:", id(greeting))
print("   same object?", id(greeting) == identity_before)
print("   value now:", repr(greeting))

print("")
print("The string method did not modify anything - it RETURNED a new")
print("string, and we rebound the name to it.")

In [ ]:
# String methods always return new strings. The original is untouched.
original = "  Python  "

# Every one of these builds a new object.
results = [
    ("strip()", original.strip()),
    ("upper()", original.upper()),
    ("replace()", original.replace("Python", "Java")),
    ("lower()", original.lower()),
]

print("original:", repr(original), "id:", id(original))
print("")
for method_name, result in results:
    print(f"   {method_name:<12} {result!r:<22} new object? {result is not original}")

print("")
print("original after all of that:", repr(original), "<- unchanged")
print("")
print("This is why a bare `text.strip()` does nothing (03.1) - you must")
print("capture the returned value.")

## Shallow immutability: the tuple trap

A tuple guarantees that its **slots** cannot be reassigned. It guarantees nothing
about the objects in those slots.

In [ ]:
# A tuple containing a list.
inner_list = [1, 2]
container = ("fixed", inner_list, 42)

print("container:", container)
print("id:", id(container))

# Replacing a slot fails - the tuple is immutable.
try:
    container[0] = "changed"
except TypeError as error:
    print("")
    print("Replacing a slot:", error)

# But mutating the list INSIDE the tuple works fine.
container[1].append(3)

print("")
print("After container[1].append(3):")
print("   container:", container, "<- the contents changed")
print("   same tuple id?", id(container))

print("")
print("The tuple still holds the same three references. One of the")
print("objects those references point at happens to have changed.")
print("")
print("Immutability is SHALLOW.")

In [ ]:
# The practical consequence: hashability.

# A tuple of immutable things can be a dict key.
safe_key = ("user", 42)
lookup = {safe_key: "works fine"}
print("Tuple of immutables as a key:", lookup[safe_key])

# A tuple containing a list cannot - it is unhashable.
unsafe_key = ("user", [1, 2])

try:
    {unsafe_key: "will fail"}
except TypeError as error:
    print("")
    print("Tuple containing a list as a key:", error)

print("")
print("WHY: hash() must be stable. If the inner list changed, the hash")
print("would change, and the dict could never find the entry again.")

# Confirm which objects are hashable.
print("")
print("Hashable?")
for candidate in [42, "text", (1, 2), frozenset([1]), [1, 2], {"a": 1}, {1, 2}]:
    try:
        hash(candidate)
        result = "yes"
    except TypeError:
        result = "no"
    print(f"   {repr(candidate):<14} {result}")

## The `+=` difference, explained properly

You met this in 04.1. Now the reason is clear: `+=` calls `__iadd__`, which
mutable types implement as an in-place change. Immutable types have no
`__iadd__`, so Python falls back to `__add__` and rebinds.

In [ ]:
# Check which types provide in-place addition.
candidates = [
    ("list", [1]),
    ("str", "a"),
    ("int", 1),
    ("tuple", (1,)),
    ("set", {1}),
    ("bytearray", bytearray(b"a")),
]

print("Type         has __iadd__?   meaning")
print("-" * 56)
for name, sample in candidates:
    has_iadd = hasattr(type(sample), "__iadd__")
    meaning = "+= mutates in place" if has_iadd else "+= rebinds to a new object"
    print(f"{name:<12} {str(has_iadd):<15} {meaning}")

print("")
print("Note: set uses __ior__ for |= rather than __iadd__.")

In [ ]:
# The difference in practice, with a shared object.

def demonstrate(label, operation):
    """Run an operation on a shared list and report what the alias sees."""
    primary = [1, 2]
    alias = primary
    identity_before = id(primary)

    primary = operation(primary)

    print(f"{label}")
    print(f"   primary: {primary}")
    print(f"   alias:   {alias}")
    print(f"   same object as before? {id(primary) == identity_before}")
    print("")


# += mutates, so the alias sees the change.
def use_iadd(values):
    values += [3]
    return values


# = + creates a new object, so the alias does not.
def use_plus(values):
    values = values + [3]
    return values


demonstrate("values += [3]", use_iadd)
demonstrate("values = values + [3]", use_plus)

print("Identical-looking code. Completely different effect on the alias.")

## Where mutability bites in real code

Four patterns worth recognising before you write them by accident.

In [ ]:
# TRAP 1: the mutable default argument (seen in 04.2, explained here).
# The default object is created ONCE, when the def line executes.

def log_event(message, history=[]):
    """Append to a default list - shared across every call."""
    history.append(message)
    return history


print("TRAP 1 - mutable default:")
print("   call 1:", log_event("started"))
print("   call 2:", log_event("stopped"), "<- both events present")
print("   the default object:", log_event.__defaults__)

# The fix.
def log_event_fixed(message, history=None):
    """Create a fresh list per call when none is supplied."""
    if history is None:
        history = []
    history.append(message)
    return history


print("")
print("   FIXED call 1:", log_event_fixed("started"))
print("   FIXED call 2:", log_event_fixed("stopped"))

In [ ]:
# TRAP 2: a mutable class attribute is shared by every instance.

class TeamBroken:
    """Every instance shares one members list."""
    members = []          # class attribute - ONE object for all instances

    def add(self, name):
        self.members.append(name)


class TeamFixed:
    """Each instance gets its own list."""

    def __init__(self):
        # Created per instance, inside __init__.
        self.members = []

    def add(self, name):
        self.members.append(name)


broken_a = TeamBroken()
broken_b = TeamBroken()
broken_a.add("Asha")

print("TRAP 2 - mutable class attribute:")
print("   team A:", broken_a.members)
print("   team B:", broken_b.members, "<- Asha appears here too")
print("   same list object?", broken_a.members is broken_b.members)

fixed_a = TeamFixed()
fixed_b = TeamFixed()
fixed_a.add("Asha")

print("")
print("   FIXED team A:", fixed_a.members)
print("   FIXED team B:", fixed_b.members)
print("   same list object?", fixed_a.members is fixed_b.members)

In [ ]:
# TRAP 3: modifying a list while looping over it.

numbers = [1, 2, 3, 4, 5, 6]

# Removing items shifts the remaining ones, so the loop skips some.
buggy = list(numbers)
for value in buggy:
    if value % 2 == 0:
        buggy.remove(value)

print("TRAP 3 - mutating during iteration:")
print("   input:  ", numbers)
print("   result: ", buggy, "<- some even numbers survived")

# The fix: build a new list rather than mutating the one you are reading.
correct = [value for value in numbers if value % 2 != 0]
print("   correct:", correct)

print("")
print("A dict raises an error instead of silently misbehaving:")
settings = {"a": 1, "b": 2}
try:
    for key in settings:
        del settings[key]
except RuntimeError as error:
    print("   ", error)

In [ ]:
# TRAP 4: a "copy" that is not a copy.

original_rows = [["a", "b"], ["c", "d"]]

# This copies the OUTER list only. The inner lists are shared.
shallow = list(original_rows)

shallow[0].append("NEW")

print("TRAP 4 - shallow copy of nested data:")
print("   original:", original_rows, "<- also changed")
print("   copy:    ", shallow)
print("   outer lists are separate?", shallow is not original_rows)
print("   inner lists are shared?  ", shallow[0] is original_rows[0])

# A deep copy duplicates everything.
import copy

original_rows = [["a", "b"], ["c", "d"]]
deep = copy.deepcopy(original_rows)
deep[0].append("NEW")

print("")
print("   with copy.deepcopy:")
print("   original:", original_rows, "<- untouched")
print("   copy:    ", deep)

print("")
print("Copying is covered fully in Chapter 11.7.")

## Choosing between them

Mutability is a design decision, not just a language fact.

In [ ]:
guidance = [
    ("Data that will change", "list / dict / set", "collecting, accumulating, updating"),
    ("A fixed record", "tuple / NamedTuple", "coordinates, database rows, RGB values"),
    ("A dictionary key", "tuple / str / int", "must be hashable"),
    ("A set member", "tuple / str / int", "must be hashable"),
    ("A default argument", "None sentinel", "never a mutable literal"),
    ("A constant collection", "tuple / frozenset", "signals it must not change"),
    ("Shared across threads", "immutable", "no locking required"),
]

print("Situation                 Use                   Why")
print("-" * 76)
for situation, use, why in guidance:
    print(situation.ljust(25), use.ljust(21), why)

print("")
print("DEFAULT: reach for immutable first. Use mutable when you genuinely")
print("need to change the object, not merely because it is familiar.")

## Takeaways

1. **Mutable** objects change in place — the identity survives. **Immutable**
   objects never change; any "modification" builds a new object.
2. The test is `id()` before and after, not whether the syntax looks like a
   modification.
3. Immutable: `int`, `float`, `bool`, `str`, `tuple`, `frozenset`, `bytes`,
   `None`. Mutable: `list`, `dict`, `set`, `bytearray`, most classes.
4. Only immutable objects are **hashable**, which is why dict keys and set
   members must be immutable.
5. **Immutability is shallow** — a tuple containing a list still lets that list
   change, and is unhashable.
6. `+=` mutates when the type has `__iadd__`, otherwise it rebinds. Same syntax,
   different effect.
7. Four traps: mutable defaults, mutable class attributes, mutating while
   iterating, and shallow copies of nested data.
8. Prefer immutable by default; choose mutable deliberately.

## Try it yourself

1. Run `test_mutability` on a custom class of your own. Is it mutable?
2. Build a tuple containing a list. Mutate the list, then try to use the tuple as
   a dict key. Explain both results.
3. Write `def f(x=[])`, call it three times, then inspect `f.__defaults__`.
4. Create a class with a mutable class attribute and two instances. Prove they
   share it with `is`.
5. Take a nested list, make a shallow copy, and find the shared inner object.